# 09 — Advanced: Observability

**Stage 9 of the workshop (Production, extended).** Tracing every model/tool call with OpenTelemetry, using the console exporter.

## Problem

Getting from "works on my laptop" to something a team can rely on — observability is what lets you see what an agent actually did in production, not just its final text output.

## Concept

Strands emits OTEL spans for every agent invocation, model call, and tool call automatically — you don't instrument your own code, you just attach an exporter. This demo uses the console exporter (prints spans to stdout, no external service, works offline). Shipping real traces to Langfuse instead is the same pattern as `model_provider.get_model()`: same agent code, only the exporter/endpoint changes.

```python
import os
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {base64_encoded_key}"
StrandsTelemetry().setup_otlp_exporter()
```

## Architecture

```
StrandsTelemetry().setup_console_exporter()   ← attached once, before any Agent runs
        │
        ▼
agent("What is 12 plus 30? Use the add tool.")
        │
        ├─ span: agent invocation
        ├─ span: model call
        ├─ span: tool call (add)
        ├─ span: model call (final answer)
        ▼
   spans printed to stdout (console exporter)
        │
        ▼
   agent's normal text result
```

## Step 1 — Attach the console exporter (before model/agent setup)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent, tool
from strands.telemetry import StrandsTelemetry

StrandsTelemetry().setup_console_exporter()

model = get_model()


## Step 2 — Define the tool and agent

`trace_attributes` attaches custom metadata to every span this agent produces.

In [ ]:
@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y


agent = Agent(model=model, tools=[add], trace_attributes={"workshop.module": "09-advanced"})


## Step 3 — Run it

OTEL spans print to stdout above the `---` separator; the agent's normal result prints after.

In [ ]:
result = agent("What is 12 plus 30? Use the add tool.")
print("---")
print(result)
